# Explainable AI (XAI) using SHAP
**Course Project:** CKD Prediction using Explainable AI (XAI)

This notebook demonstrates:
1. **Global Explainability:** Overall population biomarker importance (TreeExplainer).
2. **Local Explainability:** Patient-level risk attribution via Waterfall & Force plots.
3. **Clinical Validation:** Comparing machine-derived feature rankings against established nephrology guidelines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
from pathlib import Path
import sys

root = Path("..").resolve()
if str(root) not in sys.path:
    sys.path.append(str(root))

from src.config import SAVED_MODELS_DIR, SCALER_DIR
from src.explainability import get_explainer, compute_global_shap, explain_patient_prediction

%matplotlib inline
shap.initjs()

## 1. Load Trained Artifacts & Test Cohort

In [ ]:
model = joblib.load(SAVED_MODELS_DIR / "best_model.joblib")
test_data = joblib.load(SAVED_MODELS_DIR / "test_data.joblib")

X_test_trans = test_data["X_test_trans"]
feature_names = test_data["feature_names"]
y_test = test_data["y_test"]
print(f"Loaded model: {type(model).__name__}")
print(f"Test set features: {X_test_trans.shape[1]} across {X_test_trans.shape[0]} samples")

## 2. Global SHAP Feature Importance

In [ ]:
shap_values, importance_df = compute_global_shap(model, X_test_trans, feature_names)
importance_df.head(10)

## 3. SHAP Summary Beeswarm Plot

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_trans, feature_names=feature_names)
plt.show()

## 4. Local Patient Explanation (Waterfall Plot)

In [ ]:
# Explain the first patient in the test set
patient_sample = X_test_trans[0:1]
explanation = explain_patient_prediction(model, patient_sample, feature_names)

print(f"Predicted Probability of CKD: {explanation['prediction_probability'] * 100:.2f}%")
print(f"Diagnostic Outcome: {'CKD Detected' if explanation['is_ckd'] else 'Normal / NotCKD'}")

display(explanation['waterfall_figure'])